In [1]:
import os

import django

In [2]:
os.environ.setdefault('DJANGO_SETTINGS_MODULE', "hawc.main.settings.dev")
os.environ.setdefault('DJANGO_ALLOW_ASYNC_UNSAFE', "1")
django.setup()

In [3]:
from hawc.apps.animal.models import Endpoint
from hawc.apps.animal.exports import EndpointGroupFlatComplete
from hawc.apps.animal.serializers import EndpointSerializer

qs = Endpoint.objects.get_qs(100500085)
exporter = EndpointGroupFlatComplete(qs)
old_df = exporter.build_df()

In [6]:
old_df.head(1)

,study-id,study-hero_id,study-pubmed_id,study-doi,study-url,study-short_citation,study-full_citation,study-coi_reported,study-coi_details,study-funding_source,...,endpoint_group-response,endpoint_group-variance,endpoint_group-lower_ci,endpoint_group-upper_ci,endpoint_group-significant,endpoint_group-significance_level,endpoint_group-treatment_effect,endpoint_group-NOEL,endpoint_group-LOEL,endpoint_group-FEL
0,100518158,2543502,24418714.0,10.2131/jts.39.97,/study/100518158/,"Takahashi M et al. 2014, 2543502",Takahashi M et al. Repeated dose and reproduct...,Not reported,,This study was undertaken under the Japanese s...,...,NaN,NaN,0.013238,0.444409,False,NaN,None,False,False,False


In [24]:
len(old_df)

19364

In [7]:
from hawc.apps.common.models import sql_display, str_m2m
from hawc.apps.study.constants import CoiReported
from hawc.apps.lit.constants import ReferenceDatabase

import pandas as pd
from django.db.models import Q

In [8]:
class Module:
    def __init__(self,key_prefix:str="",query_prefix:str="",include:tuple=None,exclude:tuple=None):
        # handle include, exclude, prefix?
        self.key_prefix = key_prefix + "-" if key_prefix else key_prefix
        self.query_prefix = query_prefix + "__" if query_prefix else query_prefix
        self.include = (key_prefix+field for field in include) if include else tuple()
        self.exclude = (key_prefix+field for field in exclude) if exclude else tuple()

    @property
    def value_map(self):
        if hasattr(self,"_value_map"):
            return self._value_map
            
        value_map = self._get_value_map()
        # add key prefix
        if self.key_prefix:
            value_map = {self.key_prefix+k:v for k,v in value_map.items()}
        # add query prefix
        if self.query_prefix:
            value_map = {k:self.query_prefix+v for k,v in value_map.items()}
        # handle any includes
        if self.include:
            value_map = {k:v for k,v in value_map.items() if k in self.include}
        # handle any excludes
        if self.exclude:
            value_map = {k:v for k,v in value_map.items() if k not in self.exclude}

        
        self._value_map = value_map
        return self._value_map

    @property
    def annotation_map(self):
        if hasattr(self,"_annotation_map"):
            return self._annotation_map

        annotation_map = self._get_annotation_map(self.query_prefix)
        # add query prefix
        if self.query_prefix:
            annotation_map = {self.query_prefix+k:v for k,v in annotation_map.items()}
        # handle any includes/excludes
        if self.include or self.exclude:
            annotation_map = {k:v for k,v in annotation_map.items() if k in self.value_map.values()}

        self._annotation_map = annotation_map
        return self._annotation_map

    def _get_value_map(self):
        return {}

    def _get_annotation_map(self, query_prefix):
        return {}

    def _annotate(self,qs):
        if self.annotation_map:
            return qs.annotate(**self.annotation_map)
        return qs
        
    
    def _select_related(self,qs):
        # TODO: regex all keys in value_map, checking for greedy ending in __
        # then use that group as our select_related field ? how to handle prefetch_related ?
        # maybe remove this and just do it outside of module (ie in exporter)
        #if self.query_prefix:
        #    return qs.select_related(self.query_prefix)
        return qs

    def prepare_queryset(self,qs):
        qs = self._select_related(qs)
        qs = self._annotate(qs)
        return qs

    def prepare_df(self,df):
        # any manipulation that couldn't be handled by the ORM
        # if field missing, assume it was excluded
        # use map? something like set of fields to function?
        return df

    def get_df(self,qs):
        qs = self.prepare_queryset(qs)
        df = pd.DataFrame(data=qs.values_list(*self.value_map.values()),columns=list(self.value_map.keys()))
        return self.prepare_df(df)
        
class Exporter:
    
    def __init__(self):
        self._modules = [module[0](module[1],module[2]) for module in self.modules]

    def get_df(self,qs):
        for module in self._modules:
            qs = module.prepare_queryset(qs)
        values = [value for module in self._modules for value in module.value_map.values()]
        keys = [key for module in self._modules for key in module.value_map.keys()]
        df = pd.DataFrame(data=qs.values_list(*values),columns=keys)
        for module in self._modules:
            df = module.prepare_df(df)
        return df

In [9]:
class StudyModule(Module):
    def _get_value_map(self):
        return {
            "id":"id",
            "hero_id":"hero",
            "pubmed_id":"pmid",
            "doi":"doi",
            "url":"id", #TODO
            "short_citation":"short_citation",
            "full_citation":"full_citation",
            "coi_reported":"coi_reported",
            "coi_details":"coi_details",
            "funding_source":"funding_source",
            "bioassay":"bioassay",
            "epi":"epi",
            "epi_meta":"epi_meta",
            "in_vitro":"in_vitro",
            "eco":"eco",
            "study_identifier":"study_identifier",
            "contact_author":"contact_author",
            "ask_author":"ask_author",
            "summary":"summary",
            "editable":"editable",
            "published":"published",
        }

    def _get_annotation_map(self, query_prefix):
        return{
            "pmid": str_m2m(
                query_prefix+"identifiers__unique_id",
                filter=Q(**{query_prefix+"identifiers__database": ReferenceDatabase.PUBMED}),
            ),
            "hero": str_m2m(
                query_prefix+"identifiers__unique_id",
                filter=Q(**{query_prefix+"identifiers__database": ReferenceDatabase.HERO}),
            ),
            "doi": str_m2m(
                query_prefix+"identifiers__unique_id",
                filter=Q(**{query_prefix+"identifiers__database": ReferenceDatabase.DOI}),
            ),
            "coi_reported_display": sql_display(query_prefix+"coi_reported", CoiReported),
        }

In [10]:
from hawc.apps.study.models import Study
StudyModule().get_df(Study.objects.all()).head(1)

,id,hero_id,pubmed_id,doi,url,short_citation,full_citation,coi_reported,coi_details,funding_source,...,epi,epi_meta,in_vitro,eco,study_identifier,contact_author,ask_author,summary,editable,published
0,267758,,14630137,,267758,Lina and Kuijpers 2004,Lina BA and Kuijpers MH. Toxicity and carcinog...,0,,,...,False,False,False,False,,False,,,True,False


In [11]:
class ExperimentModule(Module):
    def _get_value_map(self):
        return {
            "id":"id",
            "url":"id", #TODO
            "name":"name",
            "type":"type",
            "has_multiple_generations":"has_multiple_generations",
            "chemical":"chemical",
            "cas":"cas",
            "dtxsid":"dtxsid",
            "chemical_source":"chemical_source",
            "purity_available":"purity_available",
            "purity_qualifier":"purity_qualifier",
            "purity":"purity",
            "vehicle":"vehicle",
            "guideline_compliance":"guideline_compliance",
            "description":"description",
        }

In [12]:
from hawc.apps.animal.models import Experiment
ExperimentModule().get_df(Experiment.objects.all()).head(1)

,id,url,name,type,has_multiple_generations,chemical,cas,dtxsid,chemical_source,purity_available,purity_qualifier,purity,vehicle,guideline_compliance,description
0,100500097,100500097,Visible Platform MWM Test,St,False,Nicotine,N/A,None,"Sigma Aldrich; St. Louis, Missouri, USA",False,,NaN,Isotonic Saline,,


In [13]:
class DosingRegimeModule(Module):
    def _get_value_map(self):
        return {
            "id":"id",
            "dosed_animals":"dosed_animals",
            "route_of_exposure":"route_of_exposure",
            "duration_exposure":"duration_exposure",
            "duration_exposure_text":"duration_exposure_text",
            "duration_observation":"duration_observation",
            "num_dose_groups":"num_dose_groups",
            "positive_control":"positive_control",
            "negative_control":"negative_control",
            "description":"description",
        }

In [14]:
from hawc.apps.animal.models import DosingRegime
DosingRegimeModule().get_df(DosingRegime.objects.all()).head(1)

,id,dosed_animals,route_of_exposure,duration_exposure,duration_exposure_text,duration_observation,num_dose_groups,positive_control,negative_control,description
0,100000456,100000557.0,OR,14.0,14 days,14.0,2,None,NR,"<p>""Mice (n = 5/group, total five groups) were..."


In [15]:
class AnimalGroupModule(Module):
    def _get_value_map(self):
        return {
            "id":"id",
            "url":"id", #TODO
            "name":"name",
            "sex":"sex",
            "animal_source":"animal_source",
            "lifestage_exposed":"lifestage_exposed",
            "lifestage_assessed":"lifestage_assessed",
            "siblings":"siblings",
            "parents":"parents",
            "generation":"generation",
            "comments":"comments",
            "diet":"diet",
            "species":"species__name",
            "strain":"strain__name",
        }

In [16]:
from hawc.apps.animal.models import AnimalGroup
AnimalGroupModule().get_df(AnimalGroup.objects.all()).head(1)

,id,url,name,sex,animal_source,lifestage_exposed,lifestage_assessed,siblings,parents,generation,comments,diet,species,strain
0,100500012,100500012,Male Sprague-Dawley Rats,M,"""Center of Experimental Animals, Chinese Acade...",Multi-lifestage,Adult,NaN,NaN,,"<p>""Healthy Sprague-Dawley rats, weighing 60–7...","""maintained on a standard feed"" - no other spe...",Rat,Sprague-Dawley


In [17]:
class EndpointModule(Module):
    def _get_value_map(self):
        return {
            "id":"id",
            "url":"id", #TODO
            "name":"name",
            "effects":"effects",
            "system":"system",
            "organ":"organ",
            "effect":"effect",
            "effect_subtype":"effect_subtype",
            "name_term_id":"name_term_id",
            "system_term_id":"system_term_id",
            "organ_term_id":"organ_term_id",
            "effect_term_id":"effect_term_id",
            "effect_subtype_term_id":"effect_subtype_term_id",
            "litter_effects":"litter_effects",
            "litter_effect_notes":"litter_effect_notes",
            "observation_time":"observation_time",
            "observation_time_units":"observation_time_units",
            "observation_time_text":"observation_time_text",
            "data_location":"data_location",
            "response_units":"response_units",
            "data_type":"data_type",
            "variance_type":"variance_type",
            "confidence_interval":"confidence_interval",
            "data_reported":"data_reported",
            "data_extracted":"data_extracted",
            "values_estimated":"values_estimated",
            "expected_adversity_direction":"expected_adversity_direction",
            "monotonicity":"monotonicity",
            "statistical_test":"statistical_test",
            "trend_value":"trend_value",
            "trend_result":"trend_result",
            "diagnostic":"diagnostic",
            "power_notes":"power_notes",
            "results_notes":"results_notes",
            "endpoint_notes":"endpoint_notes",
            "additional_fields":"additional_fields",
        }

In [18]:
from hawc.apps.animal.models import Endpoint
EndpointModule().get_df(Endpoint.objects.all()).head(1)

,id,url,name,effects,system,organ,effect,effect_subtype,name_term_id,system_term_id,...,expected_adversity_direction,monotonicity,statistical_test,trend_value,trend_result,diagnostic,power_notes,results_notes,endpoint_notes,additional_fields
0,26945,26945,4 week base excess NH4Cl male,NaN,clinical chemistry,blood,base excess,male,NaN,NaN,...,1,8,,NaN,3,,,,,{}


In [19]:
class EndpointGroupModule(Module):
    def _get_value_map(self):
        return {
            "id":"id",
            "dose_group_id":"dose_group_id",
            "n":"n",
            "incidence":"incidence",
            "response":"response",
            "variance":"variance",
            "lower_ci":"lower_ci",
            "upper_ci":"upper_ci",
            "significant":"significant",
            "significance_level":"significance_level",
            "treatment_effect":"treatment_effect",
            "NOEL":"id", #TODO
            "LOEL":"id", #TODO
            "FEL":"id", #TODO
        }

In [20]:
from hawc.apps.animal.models import EndpointGroup
EndpointGroupModule().get_df(EndpointGroup.objects.all()).head(1)

,id,dose_group_id,n,incidence,response,variance,lower_ci,upper_ci,significant,significance_level,treatment_effect,NOEL,LOEL,FEL
0,43000,0,10.0,NaN,7.2,0.6,NaN,NaN,False,NaN,NaN,43000,43000,43000


In [21]:
class EndpointExporter(Exporter):
    modules = [
        (StudyModule,"study","animal_group__experiment__study"),
        (ExperimentModule,"experiment","animal_group__experiment"),
        (AnimalGroupModule,"animal_group","animal_group"),
        (DosingRegimeModule,"dosing_regime","animal_group__dosing_regime"),
        (EndpointModule,"endpoint",""),
        (EndpointGroupModule,"endpoint_group","groups"),
    ]
    

In [22]:
from hawc.apps.animal.models import Endpoint

qs = Endpoint.objects.get_qs(100500085)
exporter = EndpointExporter()
new_df = exporter.get_df(qs)

In [27]:
new_df.head(1)

,study-id,study-hero_id,study-pubmed_id,study-doi,study-url,study-short_citation,study-full_citation,study-coi_reported,study-coi_details,study-funding_source,...,endpoint_group-response,endpoint_group-variance,endpoint_group-lower_ci,endpoint_group-upper_ci,endpoint_group-significant,endpoint_group-significance_level,endpoint_group-treatment_effect,endpoint_group-NOEL,endpoint_group-LOEL,endpoint_group-FEL
0,100518158,2543502,24418714,10.2131/jts.39.97,100518158,"Takahashi M et al. 2014, 2543502",Takahashi M et al. Repeated dose and reproduct...,3,,This study was undertaken under the Japanese s...,...,NaN,NaN,None,None,False,NaN,None,100532130.0,100532130.0,100532130.0


In [23]:
len(new_df)

19876

In [78]:
new_df.iloc[1110]

study-id                                             100519979
study-hero_id                                          3043436
study-pubmed_id                                               
study-doi                            10.1016/j.tox.2014.01.002
study-url                                            100519979
                                               ...            
endpoint_group-significance_level                          NaN
endpoint_group-treatment_effect                           None
endpoint_group-NOEL                                100533912.0
endpoint_group-LOEL                                100533912.0
endpoint_group-FEL                                 100533912.0
Name: 1110, Length: 110, dtype: object

In [79]:
new_df.iloc[1105]

study-id                                             100519979
study-hero_id                                          3043436
study-pubmed_id                                               
study-doi                            10.1016/j.tox.2014.01.002
study-url                                            100519979
                                               ...            
endpoint_group-significance_level                          NaN
endpoint_group-treatment_effect                           None
endpoint_group-NOEL                                100533912.0
endpoint_group-LOEL                                100533912.0
endpoint_group-FEL                                 100533912.0
Name: 1105, Length: 110, dtype: object

In [76]:
old_df.iloc[1110]["endpoint_group-id"]

100533917